# HydraY NNUE — v7 a 320 superbatch: la leva budget e' di nuovo viva

Runtime → **GPU (T4)**, poi "Esegui tutte". ~7h in **sedici tappe**, quindi
quasi certamente su piu' sessioni: ogni tappa lascia un checkpoint verificato su
Drive e si riparte da li'.

### Perche' ora, dopo aver detto che 320 non valeva la pena
Su **v6** il budget era saturo: i raddoppi rendevano +37,8 → +29,4 → **+9,9**, e
un altro raddoppio valeva ~+3, dentro il rumore. Quel giudizio era corretto per
v6 e basta.

v7 ha 2,97 miliardi di posizioni, quindi a 160 superbatch fa solo **5,4 epoche**
contro le 12,5 che v6 riceveva. E' meno addestrata di quanto v6 fosse quando il
budget ancora rendeva. A 320 superbatch v7 arriva a **10,8 epoche** — ancora
sotto quelle di v6.

Non e' un altro giro della stessa manovella: e' la stessa manovella su un
dataset che non l'ha mai vista girare fino in fondo.

### Cosa cambia rispetto al run a 160
Solo `TOTAL_SB`, da 160 a 320, e di conseguenza le fette girano **quattro
volte** invece di due (`1-2-3-4` ripetuto quattro volte). Stesso dataset, stessa
architettura, stesso WDL, stesso learning rate iniziale.

Il calo del learning rate si sposta da solo al superbatch 160, perche' lo
schedule keya su `TOTAL_SB`: entrambi i regimi vedono comunque tutti e quattro i
quarti.

### Il costo
Sedici tappe da 20 superbatch. Ogni tappa scompatta la sua fetta (~14 GiB da
Drive) e fa ~20 minuti di GPU. Non lasciare la scheda inattiva; quando la
sessione muore, riparti dalla cella di ripartenza con l'ultimo superbatch
salvato su Drive.

### Il riferimento da battere
La rete appena adottata: v7 a 160 superbatch, `+15,5` sulla precedente.
Startpos 48, mediogioco 930, KQvK 929, re attivi 116, loss 0,012734.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')

In [ ]:
# --- Drive + configurazione ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]

PARTS = {i: find(f'hydray_v7_part{i}.bin.zst') for i in (1, 2, 3, 4)}
# Taglia attesa DOPO la decompressione, per parte. Una decompressione
# interrotta a meta' produce un file piu' corto e nessun errore: senza questo
# controllo si addestrerebbe in silenzio su dati troncati.
RAW_SIZE = {1: 23_742_906_368, 2: 23_742_906_368,
            3: 23_742_906_368, 4: 23_745_983_264}
for i, p in PARTS.items():
    print(f'parte {i}: {os.path.getsize(p)/2**30:6.2f} GiB compressa  {p}')

NET_ID   = 'hydray-1024-v7-320sb'
TOTAL_SB = 320          # DOPPIO budget: su v7 la leva e' di nuovo viva (5,4 -> 10,8 epoche)
STAGE    = 20           # sedici tappe
ORDER    = [1, 2, 3, 4] * 4           # ogni fetta girata quattro volte
TRAINER  = '/content/th/nnue/trainer'
assert len(ORDER) * STAGE == TOTAL_SB and STAGE % 10 == 0

In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')

In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
BRANCH = 'nnue-1024'
sh('rm -rf /content/th')
sh(f'git clone --depth 1 --branch {BRANCH} https://github.com/ThomasGhione/HydraY /content/th')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 1024;' in src, 'NON e il branch a 1024 neuroni'
assert 'const INPUT_BUCKETS: usize = 4;' in src, 'i king bucket devono restare 4'
tr = open(f'{TRAINER}/src/bin/trainer.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer.rs non e a 1024'
print(f'branch {BRANCH}, 1024 neuroni, 4 king bucket: ok')

sh('apt-get -qq install -y zstd >/dev/null')
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~36 GiB (una fetta + cache di Drive)')
assert free_gb > 45, 'disco insufficiente'

In [ ]:
# --- helper delle tappe (ESEGUIRE SEMPRE, anche in ripartenza) ---

def load_slice(n):
    """Scompatta la fetta n in /content/data.bin, sostituendo la precedente."""
    if os.path.exists('/content/data.bin'):
        os.remove('/content/data.bin')          # spazio prima, non dopo
    sh(f'zstd -d -T0 --long=27 -c "{PARTS[n]}" > /content/data.bin')
    got = os.path.getsize('/content/data.bin')
    assert got == RAW_SIZE[n], f'fetta {n} troncata: {got} != {RAW_SIZE[n]}'
    print(f'fetta {n}: {got//32/1e6:.1f}M posizioni, taglia verificata', flush=True)

def stage_cmd(end, start, resume_from):
    """⚠️ STAGE_END DEVE STARE ATTACCATO A `cargo`, non in testa alla riga.
    `STAGE_END=40 cd dir && cargo ...` assegna la variabile SOLO a `cd`: cargo
    la riceve vuota, il trainer ignora le tappe e tira dritto fino a TOTAL_SB
    senza salvare niente. E' costato un run intero. Da qui l'`env` esplicito."""
    args = f'/content/data.bin {TOTAL_SB} {NET_ID}'
    if resume_from is not None:
        args += f' {start} checkpoints/{NET_ID}-{resume_from}'
    return (f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH '
            f'env CUDA_PATH=/usr/local/cuda STAGE_END={end} '
            f'cargo run -r --bin trainer --features cuda -- {args}')

def save_to_drive(end):
    """Copia il checkpoint su Drive e VERIFICA che ci sia arrivato davvero: il
    mount di Drive scrive attraverso una cache, quindi un upload mai completato
    passerebbe per riuscito."""
    ck  = f'{TRAINER}/checkpoints/{NET_ID}-{end}'
    dst = f'/content/drive/MyDrive/{NET_ID}-{end}'
    assert os.path.isdir(ck), f'checkpoint mancante in locale: {ck}'
    sh(f'rm -rf {dst} && cp -r {ck} /content/drive/MyDrive/')
    size = lambda p: sum(os.path.getsize(os.path.join(d, f))
                         for d, _, fs in os.walk(p) for f in fs)
    assert os.path.isdir(dst), f'la copia su Drive non esiste: {dst}'
    assert size(dst) == size(ck), f'copia su Drive incompleta: {size(dst)} != {size(ck)}'
    print(f'tappa fino al superbatch {end} su Drive ({size(dst)/2**20:.0f} MiB, verificata)', flush=True)

def run_stages(done=0):
    """Esegue le tappe da `done` in poi. done=0 parte da zero."""
    assert done % STAGE == 0, f'{done} non e un confine di tappa'
    prev = done if done else None
    for k in range(done // STAGE, len(ORDER)):
        end, start, sl = (k+1)*STAGE, k*STAGE + 1, ORDER[k]
        print(f'\n===== tappa {k+1}/{len(ORDER)}: superbatch {start}-{end}, fetta {sl} =====', flush=True)
        load_slice(sl)
        sh(stage_cmd(end, start, prev))
        save_to_drive(end)
        prev = end

In [ ]:
# --- training: otto tappe, fette 1-2-3-4-1-2-3-4 ---
# NON eseguire questa cella in una ripartenza: usa invece la cella in fondo.
run_stages(done=0)

In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
assert 6326288 <= sz < 6326288 + 64, f'taglia {sz}: NON e la rete a 1024'
print('quantised.bin:', sz, 'byte — 1024 neuroni confermati\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*70)
print('RIFERIMENTO — la rete adottata (v7, 2,97B, 160 SB), misurata in locale:')
print('  startpos            48      mediogioco ~24 pezzi   930')
print('  KQvK               929      KRPvKR                 110')
print('  cavallo in piu     730      re attivi (finale)     116')
print('  donna in piu      1824      training loss      0.012734')
print()
print('Stavolta la training loss E confrontabile: stesso dataset, cambia solo')
print('il budget. Deve scendere; se non scende, il budget non sta rendendo.')
print('I sanity si confrontano; il verdetto e comunque lo SPRT.')
print('='*70)

In [ ]:
# --- RIPARTENZA (usare SOLO se la sessione e' morta a meta') ---
# Come si usa:
#   1. esegui le celle da "helper" fino a "helper delle tappe" compresa;
#   2. NON eseguire la cella del training;
#   3. metti RESUME = True e DONE = ultimo superbatch salvato su Drive.
# La fetta giusta viene ricavata da ORDER: non devi ricordarti dov'era.
#
# Con RESUME = False questa cella non fa niente, cosi' "Esegui tutte" e' sicuro
# (altrimenti, a run finito, ripartirebbe da DONE rifacendo ore di training).

RESUME = False
DONE   = 20      # ultimo superbatch salvato su Drive

if not RESUME:
    print('ripartenza disattivata (RESUME = False) — nessuna azione')
else:
    ck = f'/content/drive/MyDrive/{NET_ID}-{DONE}'
    assert os.path.isdir(ck), f'checkpoint non trovato su Drive: {ck}'
    os.makedirs(f'{TRAINER}/checkpoints', exist_ok=True)
    sh(f'cp -r {ck} {TRAINER}/checkpoints/')
    assert os.path.isdir(f'{TRAINER}/checkpoints/{NET_ID}-{DONE}')
    print(f'ripartenza dal superbatch {DONE} (prossima fetta: {ORDER[DONE//STAGE]})\n')
    run_stages(done=DONE)

## Come leggere il risultato

Lo SPRT e' testa a testa contro la rete adottata, stesso binario da entrambe le
parti, candidata via `EvalFile`: l'unica differenza fisica sono i pesi, e i
pesi differiscono solo per il dataset su cui sono stati addestrati.

**Se vince** — i dati erano il vincolo, come suggeriva la curva del budget. E
si riapre subito la domanda successiva: a 5,4 epoche v7 e' probabilmente
sotto-addestrato, quindi 320 superbatch su v7 diventano interessanti proprio
mentre su v6 non lo erano.

**Se pareggia** — a parita' di costo GPU i dati in piu' non pagano, ma questo
NON chiude la questione come credeva A5: significa che il guadagno dei dati e
la perdita delle epoche si annullano. La prova decisiva sarebbe v7 a budget
alto, e costa il doppio.

**Se perde** — le 5,4 epoche non bastano, e il collo di bottiglia resta il
budget e non i dati. In quel caso il ramo giusto non e' piu' il dataset ma
l'architettura: il layer intermedio.

In tutti e tre i casi la risposta e' informativa, che e' il motivo per cui
questo run viene prima del layer.

### Una cosa da non rifare
I sanity eval **non predicono l'Elo**. La rete a 80 superbatch aveva KQvK fermo
a 657 e sembrava a corto di dati; ha poi vinto di 29,4 Elo. Guardali per
accorgerti di un disastro — mirror rotto, valori assurdi, taglia sbagliata —
non per prevedere il risultato.
